# Satisfaction probability: *how likely* is a constraint to hold?

`sat_prob_torch_mvr_chmm` computes the probability that a target constraint is satisfied, given other evidence $y$ (such as emissions) and other constraints. It returns:

$$P\big(\text{target satisfied } \;\big|\; y,\ \text{other
constraints hold}\big)$$

If the target has `time_range=[a,b]`, satisfaction is evaluated at $b$. With no `time_range`, satisfaction is evaluated at the terminal time: ie. the probability that the
whole path satisfies the target.

In [ ]:
import itertools
import math

import numpy as np
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.mvr_operators import mvr_already_satisfied
from conin.hidden_markov_model.other_queries.sat_prob_mvr import (
    sat_prob_torch_mvr_chmm,
)
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm


HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.2765440507007986,
        "B": 0.4033576072467887,
        "C": 0.32009834205241255,
    },
    transition_probs={
        ("A", "A"): 0.3391777054270445,
        ("A", "B"): 0.049711711669595204,
        ("A", "C"): 0.6111105829033604,
        ("B", "A"): 0.48102507253852517,
        ("B", "B"): 0.05601918704283972,
        ("B", "C"): 0.4629557404186351,
        ("C", "A"): 0.43616112524444134,
        ("C", "B"): 0.1773076392327265,
        ("C", "C"): 0.38653123552283214,
    },
    emission_probs={
        ("A", "lo"): 0.19949219710155375,
        ("A", "mid"): 0.30789837305397333,
        ("A", "hi"): 0.492609429844473,
        ("B", "lo"): 0.534907622618408,
        ("B", "mid"): 0.234417585356662,
        ("B", "hi"): 0.23067479202493,
        ("C", "lo"): 0.09093879934300991,
        ("C", "mid"): 0.008996844382398088,
        ("C", "hi"): 0.9000643562745919,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)


def reach_mvr(state, time_range=None, name=None):
    """MVR accepting once ``state`` has been visited; acceptance is absorbing."""
    mediation_states = ["not_yet", "seen"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("seen" if h == state else "not_yet") for h in HIDDEN_STATES},
        upd={
            (m, h): ("seen" if m == "seen" or h == state else "not_yet")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"not_yet": False, "seen": True},
        time_range=time_range,
        name=name,
    )


def forbid_mvr(state, name=None):
    """MVR rejecting any path that visits ``state``."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        name=name,
    )


print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 1. Syntax

`sat_prob_torch_mvr_chmm(model, observed, target=...)` returns a plain `float`. The
target is named by list index or by the MVR's `name`. With `return_log_weights=True`
you also get the unnormalized `[satisfied, violated]` log weights, whose `logsumexp`
is the evidence $\log P(y,\ \text{other constraints hold})$.

Checked here against two independent routes: exhaustive enumeration of all $3^7$ paths,
and the fraction of *unconstrained* FFBS draws that happen to visit `B`.

In [ ]:
reachB = MVR_CHMM(
    hidden_markov_model=hmm, constraints=[reach_mvr("B", name="reach_B")]
)

prob, log_weights = sat_prob_torch_mvr_chmm(
    reachB, observed, target="reach_B", return_log_weights=True
)


def path_weight(path):
    """P(hidden path, observations)."""
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = math.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += math.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in enumerate(observed):
        total += math.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return math.exp(total)


hit = evidence = 0.0
for path in itertools.product(HIDDEN_STATES, repeat=T):
    w = path_weight(path)
    evidence += w
    if "B" in path:
        hit += w

g = torch.Generator().manual_seed(0)
draws = ffbs_torch_mvr_chmm(
    MVR_CHMM(hidden_markov_model=hmm, constraints=[]),
    observed, num_samples=20_000, generator=g,
)

print(f"log_weights [satisfied, violated] = {np.round(log_weights.numpy(), 4)}")
print()
print(f"P(reach B | y)   sat_prob         = {prob:.6f}")
print(f"                 brute force      = {hit / evidence:.6f}")
print(f"                 FFBS (20,000)    = {np.mean(['B' in p for p in draws]):.6f}")

## 2. Every other constraint is still enforced

The target is measured; the rest condition the answer. Adding `forbid_A` removes a
whole family of routes and pushes almost all remaining mass through `B`.

In [ ]:
both = MVR_CHMM(
    hidden_markov_model=hmm,
    constraints=[forbid_mvr("A", name="forbid_A"), reach_mvr("B", name="reach_B")],
)

print(f"P(reach B | y)                = {sat_prob_torch_mvr_chmm(reachB, observed, target='reach_B'):.6f}")
print(f"P(reach B | y, forbid A)      = {sat_prob_torch_mvr_chmm(both, observed, target='reach_B'):.6f}")

## 3. `time_range` moves the evaluation point

A target windowed to `[0, b]` is initialized at `0` and evaluated at `b`, so sweeping
`b` traces out *how the probability accumulates* for this particular constraint: `reach_B`. That is, as the window lengths, we are increasingly more likely to have hit `B` at some point. Note the curves cross: `forbid_A`
starts **lower** at `b = 0`, because forbidding `A` reweights the start distribution
away from paths that begin in `B`, and only then dominates.

In [ ]:
def sat_prob_by(b, others=()):
    """P(reach B by time b), with `others` enforced."""
    model = MVR_CHMM(
        hidden_markov_model=hmm,
        constraints=[*others, reach_mvr("B", time_range=[0, b], name="reach_B")],
    )
    return sat_prob_torch_mvr_chmm(model, observed, target="reach_B")


alone = [sat_prob_by(b) for b in range(T)]
with_forbid = [sat_prob_by(b, others=[forbid_mvr("A", name="forbid_A")]) for b in range(T)]

fig, ax = plt.subplots(figsize=(8, 3.8))

ax.plot(range(T), alone, "o-", color="#2E6DB4", linewidth=2, label="reach_B alone")
ax.plot(range(T), with_forbid, "s-", color="#E08A3C", linewidth=2,
        label="reach_B, forbid_A enforced")

ax.set_xlabel("b   (window end, time_range=[0, b])")
ax.set_ylabel("P(target satisfied at b)")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(color="0.92", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print("b            :", "  ".join(f"{b:>6}" for b in range(T)))
print("alone        :", "  ".join(f"{p:>6.4f}" for p in alone))
print("+ forbid_A   :", "  ".join(f"{p:>6.4f}" for p in with_forbid))

## 4. Example: Satisfied **at time `b`** $\neq$ satisfied **ever**

`reach_B` accepts absorbingly, the two coincide for this particular constraint. However, in general the two are distinct.

Below, a parity MVR accepts on an *even* number of
visits to `B`: it flips in and out of accepting, so "holds at $T-1$" is near a coin
flip while "held at some point" is nearly certain.

In [ ]:
mediation_states = ["even", "odd"]
flip = {"even": "odd", "odd": "even"}

parity_B = HomMVR(
    hidden_states=HIDDEN_STATES,
    mediation_states=mediation_states,
    ini={h: ("odd" if h == "B" else "even") for h in HIDDEN_STATES},
    upd={
        (m, h): (flip[m] if h == "B" else m)
        for m in mediation_states
        for h in HIDDEN_STATES
    },
    evl={"even": True, "odd": False},
    name="parity_B",
)

at_end = MVR_CHMM(hidden_markov_model=hmm, constraints=[parity_B])
ever = MVR_CHMM(hidden_markov_model=hmm, constraints=[mvr_already_satisfied(parity_B)])

print(f"P(even # of B's at T-1)       = {sat_prob_torch_mvr_chmm(at_end, observed, target=0):.6f}")
print(f"P(even # of B's at some t)    = {sat_prob_torch_mvr_chmm(ever, observed, target=0):.6f}")

## Notes

- **Satisfied means `evl` holds at `b`**, the window's terminal time — the same
  definition Viterbi, FFBS and Baum-Welch enforce. It is not "satisfied at some point".
- The **"ever" variant is a composition**: `mvr_already_satisfied(target)`. There is
  deliberately no mode flag.
- `time_horizon` sets the sequence length $T$, never the evaluation point. The two
  coincide only for an unwindowed target, whose window defaults to $[0, T-1]$.
- An **unsatisfiable target is an answer, not an error** — `0.0`, with `-inf` in the
  corresponding log weight. Only infeasibility of the *other* constraints raises.
  (`sat_time_torch_mvr_chmm` raises in both cases.)